# RAG Pipelines- Data Ingestion to Vector DB Pipeline

## Data Ingestion

In [2]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
# from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

/var/folders/x8/099pvjjx5wdf3zt09d8ggq440000gn/T/ipykernel_86982/1658714240.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
/Users/piyush/Documents/code-projects/RAG-RetrievalAugmentedGeneration/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Found 4 PDF files to process

Processing: aws-overview-1-5.pdf
Loaded 5 pages

Processing: ec2-ug-1-5.pdf
Loaded 5 pages

Processing: aws-overview-105-110.pdf
Loaded 6 pages

Processing: ec2-ug-155-160.pdf
Loaded 6 pages

Total documents loaded: 22


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2026-06-19T10:46:09+00:00', 'source': '../data/pdf/aws-overview-1-5.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1', 'source_file': 'aws-overview-1-5.pdf', 'file_type': 'pdf'}, page_content='AWS Whitepaper\nOverview of Amazon Web Services\nCopyright © 2026 Amazon Web Services, Inc. and/or its aﬃliates. All rights reserved.'),
 Document(metadata={'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2026-06-19T10:46:09+00:00', 'source': '../data/pdf/aws-overview-1-5.pdf', 'total_pages': 5, 'page': 1, 'page_label': '2', 'source_file': 'aws-overview-1-5.pdf', 'file_type': 'pdf'}, page_content="Overview of Amazon Web Services AWS Whitepaper\nOverview of Amazon Web Services: AWS Whitepaper\nCopyright © 2026 Amazon Web Services, Inc. and/or its aﬃliates. All rights reserved.\nAmazon's trademarks and trade dress may not be used in connection with any product or service \n

## Chunking

In [4]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [5]:
chunks=split_documents(all_pdf_documents)
chunks

Split 22 documents into 78 chunks

Example chunk:
Content: AWS Whitepaper
Overview of Amazon Web Services
Copyright © 2026 Amazon Web Services, Inc. and/or its aﬃliates. All rights reserved....
Metadata: {'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2026-06-19T10:46:09+00:00', 'source': '../data/pdf/aws-overview-1-5.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1', 'source_file': 'aws-overview-1-5.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2026-06-19T10:46:09+00:00', 'source': '../data/pdf/aws-overview-1-5.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1', 'source_file': 'aws-overview-1-5.pdf', 'file_type': 'pdf'}, page_content='AWS Whitepaper\nOverview of Amazon Web Services\nCopyright © 2026 Amazon Web Services, Inc. and/or its aﬃliates. All rights reserved.'),
 Document(metadata={'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2026-06-19T10:46:09+00:00', 'source': '../data/pdf/aws-overview-1-5.pdf', 'total_pages': 5, 'page': 1, 'page_label': '2', 'source_file': 'aws-overview-1-5.pdf', 'file_type': 'pdf'}, page_content="Overview of Amazon Web Services AWS Whitepaper\nOverview of Amazon Web Services: AWS Whitepaper\nCopyright © 2026 Amazon Web Services, Inc. and/or its aﬃliates. All rights reserved.\nAmazon's trademarks and trade dress may not be used in connection with any product or service \n

## Embedding

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [7]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None #define ahead
        self._load_model() #protected function

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}") # default is 384 dimensions
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

#takes text(list of string) and return numpy array
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager
# running this will initialise the constructor

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6977.86it/s]


Model loaded successfully. Embedding dimension: 384


/var/folders/x8/099pvjjx5wdf3zt09d8ggq440000gn/T/ipykernel_86982/343133065.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}") # default is 384 dimensions


## VectoreStoreDB

In [8]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory) #create client that has reference to the store
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


So we have `chunks`: a list of Document objects.<br>


```Python
chunks = [
    Document(
        page_content="Machine learning is a subset of AI...",
        metadata={"page": 1}
    ),
    Document(
        page_content="Neural networks are inspired by...",
        metadata={"page": 2}
    )
]
```
We will extract all the text from the chunk and generate an embedding

Result: You are extracting only the raw text and ignoring metadata.
```Python
texts = [
    "Machine learning is a subset of AI...",
    "Neural networks are inspired by..."
]
```

### Text Extraction, Embeddings, and Vector Databases in RAG

#### Step 1: Extract Text from Chunks

```Python
    texts = []
    for doc in chunks:
        texts.append(doc.page_content)
```
#### Result

```python
    texts = [
        "Machine learning is a subset of AI...",
        "Neural networks are inspired by..."
    ]
```

At this stage, we are extracting only the raw text content from each document chunk and ignoring metadata such as page numbers and file names.
##### What is a chunk?

`chunks` is a list of `Document` objects.

For example:

```python
    chunks = [
        Document(
            page_content="Machine learning is a subset of AI...",
            metadata={"page": 1}
        ),
        Document(
            page_content="Neural networks are inspired by...",
            metadata={"page": 2}
        )
    ]  
```
---
### Step 2: Generate Embeddings

```python
    embeddings = embedding_manager.generate_embeddings(texts)
```

#### What is an Embedding?

An ```embedding``` is a numerical representation (vector) of text that captures its semantic meaning.

For example:

```python 
"cat" might become [0.23, -0.11, 0.78, ...]
```

and

```python
"kitten" might become: [0.21, -0.09, 0.81, ...]
```

Notice that the vectors are very similar because the meanings of "cat" and "kitten" are closely related.

---
### Step 3: Store in a Vector Database

```python
    vectorstore.add_documents(chunks, embeddings)
```

This stores:

```text
    Chunk Text
        +
    Embedding Vector
        +
    Metadata
```

inside a vector database such as ChromaDB or FAISS.

#### Conceptual Storage

| Chunk Text | Embedding | Metadata |
|------------|------------|------------|
| Machine learning is a subset of AI... | [0.12, 0.45, ...] | Page 1 |
| Neural networks are inspired by... | [0.98, -0.23, ...] | Page 2 |







In [9]:
chunks

[Document(metadata={'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2026-06-19T10:46:09+00:00', 'source': '../data/pdf/aws-overview-1-5.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1', 'source_file': 'aws-overview-1-5.pdf', 'file_type': 'pdf'}, page_content='AWS Whitepaper\nOverview of Amazon Web Services\nCopyright © 2026 Amazon Web Services, Inc. and/or its aﬃliates. All rights reserved.'),
 Document(metadata={'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2026-06-19T10:46:09+00:00', 'source': '../data/pdf/aws-overview-1-5.pdf', 'total_pages': 5, 'page': 1, 'page_label': '2', 'source_file': 'aws-overview-1-5.pdf', 'file_type': 'pdf'}, page_content="Overview of Amazon Web Services AWS Whitepaper\nOverview of Amazon Web Services: AWS Whitepaper\nCopyright © 2026 Amazon Web Services, Inc. and/or its aﬃliates. All rights reserved.\nAmazon's trademarks and trade dress may not be used in connection with any product or service \n

### Convert the text to Embeddings, Generate the Embeddings and, Store in the vector database

In [ ]:
### Convert the text to embeddings
texts = []

for doc in chunks:
    texts.append(doc.page_content)

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)


Generating embeddings for 78 texts...


Batches: 100%|██████████| 3/3 [00:03<00:00,  1.31s/it]

Generated embeddings with shape: (78, 384)
Adding 78 documents to vector store...
Successfully added 78 documents to vector store
Total documents in collection: 78
